# Getting Started with Flux

The code below is copied from the tutorial of Flux,
with updates necessary for later versions.

In [1]:
using Flux

## 0. a simple example

The ground truth model is defined by a matrix ``W_truth`` and vector ``b_truth``.
We aim to recover ``W_truth`` and ``b_truth``
using examples of ``ground_truth()``

In [2]:
W_truth = [1 2 3 4 5;
	       5 4 3 2 1]

2×5 Matrix{Int64}:
 1  2  3  4  5
 5  4  3  2  1

In [3]:
b_truth = [-1.0; 2.0]

2-element Vector{Float64}:
 -1.0
  2.0

In [4]:
ground_truth(x) = W_truth*x .+ b_truth

ground_truth (generic function with 1 method)

## 1. training data and the model

We generate the ground truth training data as vectors of vectors.

The training data consists of ``N`` random input vectors and as output we add noise to the ground truth evaluated at the input vectors.

In [5]:
N = 10_000

10000

In [6]:
x_train = [ 5 .* rand(5) for _ in 1:N];

In [7]:
y_train = [ ground_truth(x) + 0.2 .*randn(2) for x in x_train];

Next we define the model we want to train.

In [8]:
model(x) = W*x .+ b

model (generic function with 1 method)

As a function of ``x`` the model depends on the weight matrix ``W`` and the bias ``b``, initialized at random in the code cells below.

In [9]:
W = rand(2, 5)
b = rand(2)

2-element Vector{Float64}:
 0.6517795685475002
 0.40564389170977555

## 2. the loss function

To measure the performance, we define a loss function, which now needs to model as its first parameter.

In [10]:
"""
    function loss(M, x, y)

defines the loss for the model M,
with inputs x and outputs y.
"""
function loss(M, x, y)
    yy = M(x)
    sum(( y .- yy).^2)
end

loss

As a sanity check, we evaluate the loss function at the first instance of the training data.

In [11]:
loss(model, x_train[1], y_train[1])

3761.4469837331026

## 3. selecting the optimizer

For the optimizer, we choose the gradient descent method, which requires a value for the step size parameter.

In [12]:
opt = Descent(0.01)

Descent(0.01)

## 4. setup of the training data

In [13]:
train_data = zip(x_train, y_train);

## 5. train

For the training to apply, we must now first define a neural network model.  We have one layer, defined by ``W`` and ``b``.

In [14]:
L = Dense(W, b)

Dense(5 => 2)       # 12 parameters

In [15]:
M = Chain(L)

Chain(
  Dense(5 => 2),                        # 12 parameters
) 

The parameter collection is shown via the following.

In [16]:
# ps = Flux.params(W, b) # deprecated
ps = Flux.trainable(M)

(layers = (Dense(5 => 2),),)

To execute a training epoch, we do

In [17]:
Flux.train!(loss, M, train_data, opt)

The above command uses all of ``train_data``, which may not be necessary.  To reduce the computational effort, we can take a slice of ``train_data``, e.g.: via ``train_data = zip(x_train[1:100], y_train[1:100])`` to use the first 100 data elements.

## 6. evaluation of the training results

How well did we do?

In [18]:
@show W

W = [1.0142472445975599 2.0137615108573783 3.07110784681812 4.002482164146511 5.0039563812985755; 4.989538524769695 4.016276579899637 2.9859959678369483 2.0334734749709873 1.0518794781416378]


2×5 Matrix{Float64}:
 1.01425  2.01376  3.07111  4.00248  5.00396
 4.98954  4.01628  2.986    2.03347  1.05188

In [19]:
@show maximum(abs, W .- W_truth)

maximum(abs, W .- W_truth) = 0.0711078468181201


0.0711078468181201

## 7. monitoring the training progress

Let us reset the weight matrix ``W`` and bias ``b`` to start over and monitor the error while training.  We also need to redefine the model, now called ``M2``.

In [20]:
W = rand(2, 5)
b = rand(2)
M2 = Chain(Dense(W, b))

Chain(
  Dense(5 => 2),                        # 12 parameters
) 

The training epoch needs to be able to update the ``state``, defined in the code cell below.

In [21]:
state = Flux.setup(opt, M2)

(layers = ((weight = Leaf(Descent(0.01), nothing), bias = Leaf(Descent(0.01), nothing), σ = ()),),)

Executing a training epoch as below, we monitor the loss.

In [22]:
nbr = 100
for (x, y) in zip(x_train[1:nbr], y_train[1:nbr])
    gs = gradient(m -> loss(m, x, y), M2) 
    println(loss(M2, x, y))
    Flux.Optimise.update!(state, M2, gs[1])
end

4567.357304361625
553.245833285954
483.495210214943
12.449436308190595
285.6257320848692
206.76473841580028
282.4342036989557
35.935431394062434
224.30710674030985
48.27349434845312
19.330243030236904
42.10185758891386
3.6958737236429444
12.573886025896503
2.0824167640130886
34.4195301641794
38.55261510067439
6.005866000314168
5.506868982119332
26.832612337169476
30.95632570874156
16.046621288643045
0.823195196270534
1.2105549290506756
21.47650651942783
36.41300146343701
12.110637753848968
1.7286918960391102
2.759415237770284
9.166914904948518
1.497946334433237
6.748319003653388
2.5204779502019026
3.260907795339238
10.656263799233962
2.6453086761731655
6.842529146203612
7.089395986635256
2.250954564150082
12.41610177120226
8.056877542810183
4.575267190465548
1.9493420824419512
4.787602390511638
0.9176100046569426
0.36186173827002016
0.640414329363628
1.0837171435532458
1.0361860533866787
0.7080788619274915
0.23898593871976204
1.1999973245448856
0.23682430983194908
0.6132342393537717
2.

In [23]:
@show W

W = [0.7651632087278496 1.9307594478285286 2.8159979332867633 3.945582632282975 4.817345349709885; 5.141966680191063 4.000216447628437 3.032893106736955 2.0786753375912923 1.1295692523306626]


2×5 Matrix{Float64}:
 0.765163  1.93076  2.816    3.94558  4.81735
 5.14197   4.00022  3.03289  2.07868  1.12957

In [24]:
@show maximum(abs, W .- W_truth)

maximum(abs, W .- W_truth) = 0.2348367912721504


0.2348367912721504

Using the first 100 elements of the training data, we get only about one correct decimal place.